# Gene panel heatmaps

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

# Resolve this analysis folder (paper/01_rna) regardless of the kernel's cwd.
def _here():
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / '00_build_metadata.py').exists():
            return c
    raise RuntimeError('run this notebook from inside paper/01_rna')

DIR = _here()

TF_LIST = DIR / 'data' / 'genes_tfs.txt'
MARKER_LIST = DIR / 'data' / 'genes_markers.txt'
GETMM_PARQUET = DIR / 'results' / 'gene_getmm_log2.parquet'   # <- 12_normalize_getmm.R

PNG_DIR = DIR / 'figs'
PDF_DIR = Path(os.environ['FIGURE_PDF_DIR']) if os.environ.get('FIGURE_PDF_DIR') else PNG_DIR
PNG_DIR.mkdir(parents=True, exist_ok=True)

STAGES = ['ESC', 'DE', 'HB', 'iHEP', 'mHEP']
CELL = 0.3      # inches per heatmap cell
CMAP = 'cividis'

# RefSeq -> GENCODE gene-symbol updates (applied to the curated lists; extend as needed)
RENAME = {'CTGF': 'CCN2', 'CYR61': 'CCN1'}

In [ ]:
def parse_marker_list(path):
    sections, name, cur = [], '(no section)', []
    for raw in path.read_text().splitlines():
        s = raw.strip()
        if not s:
            continue
        if s.startswith('#'):
            if cur:
                sections.append((name, cur)); cur = []
            name = s.lstrip('#').strip().strip('=').strip()
        else:
            g = s.split('#', 1)[0].strip()
            if g:
                cur.append(g)
    if cur:
        sections.append((name, cur))
    return sections


def remap(genes):
    return [RENAME.get(g, g) for g in genes]


getmm = pl.read_parquet(GETMM_PARQUET).unique(subset='gene_name', keep='first')
rna_set = set(getmm['gene_name'].to_list())
sample_cols = [c for c in getmm.columns if any(c.startswith(f'{s}_REP') for s in STAGES)]


def stage_getmm_for(genes):
    pdf = (getmm.filter(pl.col('gene_name').is_in(genes))
              .select(['gene_name'] + sample_cols).to_pandas()
              .set_index('gene_name').loc[genes])
    return pd.DataFrame({s: pdf[[f'{s}_REP1', f'{s}_REP2']].mean(axis=1) for s in STAGES})


# TFs
tfs = remap(list(dict.fromkeys(t.strip() for t in TF_LIST.read_text().splitlines() if t.strip())))
tf_present = [t for t in tfs if t in rna_set]
print('TFs:', len(tf_present), 'present /', len(tfs), 'requested;',
      'absent:', [t for t in tfs if t not in rna_set])
tf_g = stage_getmm_for(tf_present)

# Stage markers
sections = parse_marker_list(MARKER_LIST)
pruned, ordered = [], []
for nm, gs in sections:
    kept = [g for g in remap(gs) if g in rna_set and g not in ordered]
    if kept:
        pruned.append((nm, kept)); ordered.extend(kept)
mk_g = stage_getmm_for(ordered)
bounds = np.cumsum([0] + [len(gs) for _, gs in pruned])
print('markers:', len(ordered), 'in', len(pruned), 'sections')

## Plot

In [ ]:
# shared color scale across BOTH panels (vmin fixed at 0; log GeTMM >= 0)
VMAX = float(max(tf_g.values.max(), mk_g.values.max()))
print('shared log-GeTMM scale: vmin=0  vmax=%.2f' % VMAX)

nT, nM = len(tf_present), len(ordered)
fig = plt.figure(figsize=(9, max(nT, nM) * CELL + 1.3))
gs = fig.add_gridspec(1, 5, width_ratios=[5, 2.4, 5, 0.5, 0.3], wspace=0.05)
axT = fig.add_subplot(gs[0, 0])
axM = fig.add_subplot(gs[0, 2])
axS = fig.add_subplot(gs[0, 3])
axC = fig.add_subplot(gs[0, 4])

for ax, data, names, title in [(axT, tf_g, tf_present, 'Curated TFs'),
                               (axM, mk_g, ordered, 'Stage markers')]:
    im = ax.imshow(data.values, aspect='equal', cmap=CMAP, vmin=0, vmax=VMAX)
    ax.set_anchor('N')
    ax.set_xticks(range(len(STAGES))); ax.set_xticklabels(STAGES, fontsize=9, rotation=90)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
    ax.tick_params(length=0, pad=3)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_title(title, fontsize=11)

# TF panel section dividers — after pluripotency (SOX2) and DE specifiers (SOX17)
for g in ('SOX2', 'SOX17'):
    if g in tf_present:
        axT.axhline(tf_present.index(g) + 0.5, color='white', lw=1.2)

# markers section dividers + labels in the markers-panel coords (track aspect-equal rows)
for b in bounds[1:-1]:
    axM.axhline(b - 0.5, color='white', lw=1.2)
axS.axis('off')
for (nm, gs_), a, b in zip(pruned, bounds[:-1], bounds[1:]):
    axM.text(len(STAGES) - 0.35, (a + b - 1) / 2, nm, ha='left', va='center',
             rotation=90, fontsize=8, fontstyle='italic', clip_on=False)

cb = fig.colorbar(im, cax=axC); cb.set_label('log GeTMM', fontsize=9); cb.ax.tick_params(labelsize=8)
fig.suptitle('Curated TFs & stage markers — RNA log GeTMM (shared scale)', fontsize=12, y=0.99)

fig.savefig(DIR / 'figs' / 'tf_stage_markers_getmm.png', dpi=200, bbox_inches='tight')
fig.savefig(DIR / 'figs' / 'tf_stage_markers_getmm.pdf', bbox_inches='tight');